# 03: STRING Networks for pLLPS-Enriched Functional Groups

This notebook fetches STRING protein-protein interactions for functionally enriched membrane protein groups and enriches them with pLLPS scores.

**Workflow:**
1. Load pLLPS-enriched functional groups from notebook 02
2. For each enriched group, fetch high-confidence STRING interactions
3. Filter for high-confidence interactions (score ≥ 700/1000)
4. Match interactors with pLLPS scores
5. Save interaction data for visualization

**Inputs:**
- `results/pllps_enriched_groups.json`: List of enriched groups
- `results/functional_groups_with_pllps.csv`: Classified proteins with scores
- `results/full_dataset.csv`: Full dataset with pLLPS scores

**Outputs:**
- `results/string_networks_by_group/{group}_interactions.csv`: Interactions per group
- `results/enriched_group_interactions_summary.json`: Summary statistics

In [3]:
# Import required libraries
import pandas as pd
import numpy as np
import llps_functions as lf
import json
import importlib
from pathlib import Path

# Reload module
importlib.reload(lf)

# Constants
SCORE_THRESHOLD = 700  # STRING score threshold (0-1000 scale)
STRING_CACHE_DIR = Path('data/string_cache')
STRING_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Libraries imported successfully")
print(f"STRING score threshold: {SCORE_THRESHOLD}/1000")

✅ Libraries imported successfully
STRING score threshold: 700/1000


## 1. Load Enriched Functional Groups and pLLPS Data

In [4]:
# Load data
df_full = lf.load_analysis_result('full_dataset', format='csv')
functional_groups_df = lf.load_analysis_result('functional_groups_with_pllps', format='csv')

# Load enriched groups list
with open('results/pllps_enriched_groups.json', 'r') as f:
    enriched_data = json.load(f)
    enriched_groups = enriched_data.get('enriched_groups', enriched_data.get('high_enriched', []))

print(f"\n📊 Data Loaded:")
print(f"   Total proteins with pLLPS: {len(df_full)}")
print(f"   Classified membrane proteins: {len(functional_groups_df)}")
print(f"   Enriched groups identified: {len(enriched_groups)}")
print(f"\n   Enriched groups: {enriched_groups}")

# Create pLLPS lookup dictionary
pllps_dict = dict(zip(df_full['Entry'], df_full['p(LLPS)']))
print(f"\n   pLLPS lookup created: {len(pllps_dict)} proteins")

✅ Loaded CSV from: results/full_dataset.csv (20366 rows)
✅ Loaded CSV from: results/functional_groups_with_pllps.csv (20366 rows)

📊 Data Loaded:
   Total proteins with pLLPS: 20366
   Classified membrane proteins: 20366
   Enriched groups identified: 6

   Enriched groups: ['Structural', 'Other', 'Enzyme', 'Ion Channel', 'Transporter', 'GPCR']

   pLLPS lookup created: 20366 proteins


## 2. Fetch and Process STRING Interactions for Each Group

In [ ]:
# Process each enriched group
output_dir = Path('results/string_networks_by_group')
output_dir.mkdir(parents=True, exist_ok=True)

all_interactions_summary = {}

# Filter to reasonable-sized groups for STRING analysis
# (Excluding 'Other' [15393] and 'Enzyme' [2840] - too large for comprehensive analysis)
groups_to_process = ['Structural', 'Ion Channel', 'Transporter', 'GPCR']
enriched_groups_filtered = [g for g in enriched_groups if g in groups_to_process]
print(f"\n⚠️  Processing selected groups with manageable sizes:")
print(f"   Groups: {enriched_groups_filtered}")
print(f"   (Skipping 'Other' [n=15393] and 'Enzyme' [n=2840] due to size)\\n")

for group_name in enriched_groups_filtered:
    print(f"\n{'='*60}")
    print(f"Processing: {group_name}")
    print(f"{'='*60}")
    
    # Get proteins in this group
    group_proteins = functional_groups_df[functional_groups_df['Functional Group'] == group_name].copy()
    protein_ids = group_proteins['Entry'].tolist()
    
    print(f"\n📥 Fetching interactions for {len(protein_ids)} proteins...")
    
    # Fetch interactions using llps_functions
    all_interactions = []
    high_score_interactions = []
    
    for protein_id in protein_ids:
        try:
            interactions = lf.fetch_string_interactions(protein_id, STRING_CACHE_DIR)
            all_interactions.extend(interactions)
        except Exception as e:
            print(f"   ⚠️  Error fetching {protein_id}: {str(e)[:50]}")
    
    if not all_interactions:
        print(f"   ⚠️  No interactions found for {group_name}")
        all_interactions_summary[group_name] = {'count': 0, 'high_score': 0}
        continue
    
    # Convert to DataFrame
    interactions_df = pd.DataFrame(all_interactions)
    
    # Filter for high-confidence interactions
    high_score_mask = interactions_df['combined_score'] >= SCORE_THRESHOLD
    interactions_df = interactions_df[high_score_mask].copy()
    
    print(f"\n✓ Total interactions: {len(interactions_df)}")
    
    if interactions_df.empty:
        print(f"   ⚠️  No high-confidence interactions found")
        all_interactions_summary[group_name] = {'count': 0, 'high_score': 0}
        continue
    
    # Match with pLLPS scores
    interactions_df = lf.match_interactions_to_pllps(interactions_df, pllps_dict)
    
    # Calculate summary statistics
    high_high = (interactions_df['partner_pllps'] > 0.7).sum()
    high_low = ((interactions_df['partner_pllps'] <= 0.7) & (interactions_df['partner_pllps'].notna())).sum()
    
    print(f"   High-High interactions (both >0.7): {high_high}")
    print(f"   High-Low interactions (one >0.7): {high_low}")
    print(f"   Mean partner pLLPS: {interactions_df['partner_pllps'].mean():.3f}")
    
    # Save to CSV
    output_file = output_dir / f"{group_name.lower().replace(' ', '_')}_interactions.csv"
    interactions_df.to_csv(output_file, index=False)
    print(f"\n✅ Saved: {output_file}")
    
    # Add to summary
    all_interactions_summary[group_name] = {
        'protein_count': len(protein_ids),
        'interaction_count': len(interactions_df),
        'high_high_interactions': high_high,
        'high_low_interactions': high_low,
        'mean_partner_pllps': float(interactions_df['partner_pllps'].mean()),
        'file': str(output_file)
    }

# Save summary
summary_file = Path('results/enriched_group_interactions_summary.json')
with open(summary_file, 'w') as f:
    json.dump(all_interactions_summary, f, indent=2)

print(f"\n{'='*60}")
print(f"✅ Summary saved: {summary_file}")
print(f"\nReady for network visualization in notebook 04!")